In [4]:
# 1. Import Libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch
import time

In [5]:
# 2. Load Cleaned Data
df_resume = pd.read_csv("../data/processed/resumes_cleaned.csv")
df_jd = pd.read_csv("../data/processed/jd_cleaned.csv")

# Untuk simulasi cepat, kita ambil 500 resume dan 100 JD
# (Proses TF-IDF sangat cepat, tapi SentenceTransformer butuh waktu)
resumes_subset = df_resume["cleaned_text"].dropna().head(500).tolist()
jd_subset = df_jd["cleaned_text"].dropna().head(100).tolist()

print(f"Loaded {len(resumes_subset)} resumes and {len(jd_subset)} job descriptions for testing.")

Loaded 500 resumes and 100 job descriptions for testing.


## Pendekatan 1: TF-IDF (Term Frequency-Inverse Document Frequency)

Mewakili keahlian **`scikit-learn`**.
TF-IDF mencari kata yang unik di dalam sebuah resume dibandingkan dengan keseluruhan dokumen.

In [6]:
# 3. Feature Extraction dengan TF-IDF
print("Training TF-IDF Vectorizer...")
start_time = time.time()

tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Ambil 5000 kata paling penting

# Fit ke seluruh data resume
tfidf_resume_features = tfidf_vectorizer.fit_transform(resumes_subset)

# Transform JD menggunakan vectorizer yang sama
tfidf_jd_features = tfidf_vectorizer.transform(jd_subset)

print(f"TF-IDF extraction took {time.time() - start_time:.2f} seconds.")
print(f"Resume features shape: {tfidf_resume_features.shape}")

Training TF-IDF Vectorizer...
TF-IDF extraction took 1.04 seconds.
Resume features shape: (500, 5000)


In [7]:
# 4. Mencari Kecocokan (Match) dengan TF-IDF Cosine Similarity
# Bandingkan JD pertama dengan seluruh resume
jd_index = 0
tfidf_similarities = cosine_similarity(tfidf_jd_features[jd_index], tfidf_resume_features)[0]

# Ambil indeks 3 resume terbaik
top_3_tfidf_idx = tfidf_similarities.argsort()[-3:][::-1]

print(f"--- HASIL MATCHING TF-IDF UNTUK JD ID {jd_index} ---")
for idx in top_3_tfidf_idx:
    print(f"Resume ID: {idx} | Similarity Score: {tfidf_similarities[idx]:.4f}")

--- HASIL MATCHING TF-IDF UNTUK JD ID 0 ---
Resume ID: 331 | Similarity Score: 0.1907
Resume ID: 471 | Similarity Score: 0.1706
Resume ID: 293 | Similarity Score: 0.1595


## Pendekatan 2: Sentence Transformers (Semantic Embedding)

Mewakili keahlian **`PyTorch`** dan **`NLP`** modern.
Embedding mampu memahami *makna* teks (misal: 'machine learning' dianggap mirip dengan 'artificial intelligence' walaupun ejaannya beda).

In [8]:
# 5. Feature Extraction dengan Sentence Transformers
print("Loading SentenceTransformer Model (menggunakan PyTorch backend)...")
# Cek apakah GPU tersedia
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Model ringan 'all-MiniLM-L6-v2' ideal untuk semantic search
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

start_time = time.time()
print("Generating Embeddings for Resumes...")
embedding_resume_features = model.encode(resumes_subset, show_progress_bar=True)

print("Generating Embeddings for Job Descriptions...")
embedding_jd_features = model.encode(jd_subset, show_progress_bar=True)

print(f"Embedding extraction took {time.time() - start_time:.2f} seconds.")
print(f"Resume embeddings shape: {embedding_resume_features.shape}")

Loading SentenceTransformer Model (menggunakan PyTorch backend)...
Using device: cpu


d:\Project\01 Skills\AI-Resume-Screener\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Willy Hanafi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1801.48it/s]


Generating Embeddings for Resumes...


Batches: 100%|██████████| 16/16 [00:42<00:00,  2.65s/it]


Generating Embeddings for Job Descriptions...


Batches: 100%|██████████| 4/4 [00:08<00:00,  2.09s/it]

Embedding extraction took 50.78 seconds.
Resume embeddings shape: (500, 384)


In [9]:
# 6. Mencari Kecocokan (Match) dengan Semantic Cosine Similarity
# Bandingkan JD pertama dengan seluruh resume (menggunakan Tensor)
jd_tensor = torch.tensor(embedding_jd_features[jd_index]).unsqueeze(0)
resume_tensors = torch.tensor(embedding_resume_features)

# Hitung Cosine Similarity menggunakan PyTorch
semantic_similarities = torch.nn.functional.cosine_similarity(jd_tensor, resume_tensors).numpy()

top_3_semantic_idx = semantic_similarities.argsort()[-3:][::-1]

print(f"--- HASIL MATCHING SEMANTIC (TRANSFORMERS) UNTUK JD ID {jd_index} ---")
for idx in top_3_semantic_idx:
    print(f"Resume ID: {idx} | Similarity Score: {semantic_similarities[idx]:.4f}")

--- HASIL MATCHING SEMANTIC (TRANSFORMERS) UNTUK JD ID 0 ---
Resume ID: 316 | Similarity Score: 0.6423
Resume ID: 331 | Similarity Score: 0.6274
Resume ID: 465 | Similarity Score: 0.6169
